# Colab CUDA Training

This notebook is a single-file Colab entrypoint for training with CUDA, saving checkpoints to Google Drive, and resuming automatically from `latest_checkpoint.pt`.

## What This Notebook Does

- clones the repo into `/content/cs286-project`
- mounts Google Drive
- checks that a CUDA GPU is available
- launches `supervised`, `contrastive`, or `probe` training
- saves artifacts under `MyDrive/cs286-project-runs/<stage>/<run-name>/`
- resumes automatically if `latest_checkpoint.pt` already exists

In [ ]:
REPO_URL = "https://github.com/your-user/your-repo.git"
PROJECT_ROOT = "/content/cs286-project"
DRIVE_ROOT = "/content/drive/MyDrive/cs286-project-runs"

STAGE = "supervised"  # one of: supervised, contrastive, probe
RUN_NAME = "fusion-full"
EXPERIMENT_NAME = "fusion_full_colab"
SAVE_EVERY = 1
FORCE_FRESH_RUN = False

# Edit these per stage. For probe runs, include --encoder-ckpt-path.
RUNNER_ARGS = {
    "supervised": [
        "--channel-mode", "fusion",
        "--label-fraction", "1.0",
        "--epochs", "40",
        "--patience", "7",
        "--batch-size", "256",
    ],
    "contrastive": [
        "--epochs", "40",
        "--patience", "7",
        "--batch-size", "256",
    ],
    "probe": [
        "--encoder-ckpt-path", f"{DRIVE_ROOT}/contrastive/pretrain/latest_checkpoint.pt",
        "--evaluation-mode", "pair",
        "--label-fraction", "0.1",
        "--epochs", "40",
        "--patience", "7",
        "--batch-size", "256",
    ],
}

assert STAGE in RUNNER_ARGS, f"Unsupported stage: {STAGE}"
print({
    "stage": STAGE,
    "run_name": RUN_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "drive_root": DRIVE_ROOT,
    "runner_args": RUNNER_ARGS[STAGE],
})

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

project_root = Path(PROJECT_ROOT)
if project_root.exists():
    print(f"Using existing repo at {project_root}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"], check=True)
print("Setup complete")

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print(f"Drive output root: {DRIVE_ROOT}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. In Colab, go to Runtime > Change runtime type > GPU.")

print("CUDA available")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

output_dir = Path(DRIVE_ROOT) / STAGE / RUN_NAME
output_dir.mkdir(parents=True, exist_ok=True)
latest_checkpoint_path = output_dir / "latest_checkpoint.pt"

command = [
    sys.executable,
    "-m",
    "src.train_colab",
    "--stage", STAGE,
    "--project-root", PROJECT_ROOT,
    "--drive-root", DRIVE_ROOT,
    "--run-name", RUN_NAME,
    "--experiment-name", EXPERIMENT_NAME,
    "--save-every", str(SAVE_EVERY),
    "--",
    *RUNNER_ARGS[STAGE],
]

if FORCE_FRESH_RUN:
    command.insert(-len(RUNNER_ARGS[STAGE]) - 1, "--no-resume")

print("Command:")
print(" ".join(shlex.quote(part) for part in command))
print(f"Output dir: {output_dir}")
print(f"Latest checkpoint: {latest_checkpoint_path}")

subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
from pathlib import Path
import json

output_dir = Path(DRIVE_ROOT) / STAGE / RUN_NAME
metrics_paths = sorted(output_dir.glob("*_metrics.json"))
checkpoint_paths = sorted(output_dir.glob("*_checkpoint.pt"))

print("Metrics files:")
for path in metrics_paths:
    print(" -", path)

print("\nCheckpoint files:")
for path in checkpoint_paths:
    print(" -", path)

print("\nLatest checkpoint:")
print(output_dir / "latest_checkpoint.pt")

if metrics_paths:
    latest_metrics = json.loads(metrics_paths[-1].read_text())
    print("\nMetrics summary:")
    print(json.dumps({
        "experiment_name": latest_metrics.get("experiment_name"),
        "best_epoch": latest_metrics.get("best_epoch"),
        "artifacts": latest_metrics.get("artifacts"),
        "train": latest_metrics.get("train"),
        "val": latest_metrics.get("val"),
        "test": latest_metrics.get("test"),
    }, indent=2))